In [0]:
trips_df = spark.table("hyf.nyc_yellow.raw_trips")
zones_df = spark.table("hyf.nyc_yellow.raw_zones")

trips_df.printSchema()
zones_df.printSchema()

root
 |-- vendor_id: long (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

root
 |-- location_id: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |--

In [0]:
from pyspark.sql import functions as F

pickup_borough_counts = (
    trips_df
    .join(
        zones_df,
        trips_df.pickup_location_id == zones_df.location_id,
        "left"
    )
    .groupBy("borough")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.desc("trip_count"))
)

pickup_borough_counts.show()

+-------------+----------+
|      borough|trip_count|
+-------------+----------+
|    Manhattan| 112028489|
|       Queens|  12292035|
|     Brooklyn|   2648890|
|        Bronx|    570457|
|      Unknown|    559802|
|          N/A|     76659|
|          EWR|     17103|
|Staten Island|      9113|
+-------------+----------+



In [0]:
average_amount_by_payment = (
    trips_df
    .groupBy("payment_type")
    .agg(
        F.avg("total_amount").alias("avg_total_amount")
    )
    .orderBy("payment_type")
)

average_amount_by_payment.show()

+------------+------------------+
|payment_type|  avg_total_amount|
+------------+------------------+
|           0| 23.48503280062518|
|           1|29.999458495708435|
|           2| 23.74697069224019|
|           3| 9.028713996517435|
|           4|2.1613990066710844|
|           5|14.887777777777778|
+------------+------------------+



## PySpark vs dbt SQL

I would use dbt SQL for SQL-based transformations, data marts, and analytics models because it is simpler and easier to maintain. I would use PySpark when I need Python code, large-scale distributed processing, or functionality that cannot be expressed well in SQL, such as machine learning or external API integration.